# Chicago Housing Affordability Analysis (Updated)

## Core metrics
- **PIR** (Payment-to-Income Ratio): monthly mortgage payment / monthly household income
- **PTI** (Price-to-Income Ratio): anchored home price / annual household income
- **API** (Affordability Pressure Index): simplified pressure proxy (auxiliary only)

> Note: We use an **anchored-price method** because median transaction price is unavailable in the current dataset.


In [1]:
import os
import numpy as np
import pandas as pd
import altair as alt
from sklearn.linear_model import LinearRegression

os.makedirs('output', exist_ok=True)


In [2]:
# Data sources
urls = {
    'home_price': 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=CHXRSA',
    'mortgage_rate': 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US',
    'income_median': 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=MHIIL17000A052NCEN',
    'income_pc_real': 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=RPIPC16980',
    'permits': 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=CHIC917BPPRIVSA'
}

home = pd.read_csv(urls['home_price']).rename(columns={'observation_date':'date','CHXRSA':'home_price_index'})
mort = pd.read_csv(urls['mortgage_rate']).rename(columns={'observation_date':'date','MORTGAGE30US':'mortgage_rate'})
income_med = pd.read_csv(urls['income_median']).rename(columns={'observation_date':'date','MHIIL17000A052NCEN':'income_median'})
income_pc = pd.read_csv(urls['income_pc_real']).rename(columns={'observation_date':'date','RPIPC16980':'income_pc_real'})
permits = pd.read_csv(urls['permits']).rename(columns={'observation_date':'date','CHIC917BPPRIVSA':'permits'})

for df in [home,mort,income_med,income_pc,permits]:
    df['date']=pd.to_datetime(df['date'])


In [3]:
# Annualize and merge
start_date='2000-01-01'
home=home[home['date']>=start_date]
mort=mort[mort['date']>=start_date]
permits=permits[permits['date']>=start_date]
income_med=income_med[income_med['date']>=start_date]
income_pc=income_pc[income_pc['date']>=start_date]

home_y=home.set_index('date').resample('Y').mean().reset_index()
mort_y=mort.set_index('date').resample('Y').mean().reset_index()
permits_y=permits.set_index('date').resample('Y').mean().reset_index()

income_med['year']=income_med['date'].dt.year
income_pc['year']=income_pc['date'].dt.year
home_y['year']=home_y['date'].dt.year
mort_y['year']=mort_y['date'].dt.year
permits_y['year']=permits_y['date'].dt.year

merged=(home_y[['year','home_price_index']]
        .merge(mort_y[['year','mortgage_rate']],on='year',how='inner')
        .merge(income_med[['year','income_median']],on='year',how='inner')
        .merge(income_pc[['year','income_pc_real']],on='year',how='left')
        .merge(permits_y[['year','permits']],on='year',how='left')
        .dropna(subset=['home_price_index','mortgage_rate','income_median']))

merged=merged.sort_values('year').reset_index(drop=True)


/var/folders/__/fs52ryv53yzdbmtq_mns4z5c0000gn/T/ipykernel_25294/727683234.py:9: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  home_y=home.set_index('date').resample('Y').mean().reset_index()
/var/folders/__/fs52ryv53yzdbmtq_mns4z5c0000gn/T/ipykernel_25294/727683234.py:10: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  mort_y=mort.set_index('date').resample('Y').mean().reset_index()
/var/folders/__/fs52ryv53yzdbmtq_mns4z5c0000gn/T/ipykernel_25294/727683234.py:11: FutureWarning: 'Y' is deprecated and will be removed in a future version, please use 'YE' instead.
  permits_y=permits.set_index('date').resample('Y').mean().reset_index()


In [4]:
merged

,year,home_price_index,mortgage_rate,income_median,income_pc_real,permits
0,2000,104.956584,8.053462,46327.0,NaN,3422.590194
1,2001,113.447075,6.967885,46991.0,NaN,3648.453703
2,2002,121.615631,6.537308,44946.0,NaN,3872.334938
3,2003,131.714450,5.826981,47367.0,NaN,4149.642724
4,2004,142.885084,5.839231,47711.0,NaN,3977.264666
5,2005,156.339651,5.866731,50270.0,NaN,4381.036571
6,2006,166.654735,6.413269,52012.0,NaN,3897.268800
7,2007,164.950865,6.337308,54141.0,NaN,2857.993064
8,2008,148.410472,6.027170,56230.0,49533.0,1391.381581
9,2009,127.357482,5.036538,53974.0,46053.0,555.567120


In [5]:
# Build affordability metrics
# Anchored price proxy: choose baseline nominal price at first available year
P_BASE = 320_000
DOWN_PAYMENT = 0.20
N_MONTHS = 360

base_hpi = merged.loc[0,'home_price_index']
merged['home_price_proxy'] = P_BASE * (merged['home_price_index']/base_hpi)
merged['loan_amount'] = merged['home_price_proxy']*(1-DOWN_PAYMENT)
merged['r_month'] = (merged['mortgage_rate']/100)/12

# Mortgage payment formula
merged['mortgage_payment_monthly'] = merged['loan_amount'] * (
    merged['r_month']*(1+merged['r_month'])**N_MONTHS / ((1+merged['r_month'])**N_MONTHS - 1)
)

merged['income_monthly'] = merged['income_median']/12
# Payment-to-Income Ratio
merged['PIR'] = merged['mortgage_payment_monthly']/merged['income_monthly']
# Price-to-income Ratio
merged['PTI'] = merged['home_price_proxy']/merged['income_median']

####可能删掉 Keep old simplified metric as auxiliary API
merged['API'] = merged['home_price_index'] * (merged['mortgage_rate']/100) / merged['income_pc_real']

# Normalized index
base_pir=merged.loc[0,'PIR']
merged['PIR_index']=merged['PIR']/base_pir*100


In [6]:
# Chart 1: trend overview (normalized)
plot_df=merged[['year','home_price_index','mortgage_rate','income_median']].copy()
for c in ['home_price_index','mortgage_rate','income_median']:
    plot_df[c+'_idx']=plot_df[c]/plot_df[c].iloc[0]*100

plot_long=plot_df.melt(id_vars=['year'],value_vars=['home_price_index_idx','mortgage_rate_idx','income_median_idx'],var_name='series',value_name='value')
plot_long['series']=plot_long['series'].map({'home_price_index_idx':'Home Price Index (norm)','mortgage_rate_idx':'Mortgage Rate (norm)','income_median_idx':'Median Income (norm)'})
chart1=alt.Chart(plot_long).mark_line().encode(x='year',y='value',color='series').properties(title='Chicago Housing Affordability Drivers (Normalized to 100)',width=500,height=250)
chart1.save('output/trend_overview_normalized.png')
chart1


alt.Chart(...)

In [14]:
# Chart 2: PIR + PIR index
base=alt.Chart(merged)
left=base.mark_line(color='#1f77b4').encode(alt.X('year'),alt.Y('PIR',title='PIR'),color=alt.value('#1f77b4')).properties(width=500,height=250)
right=base.mark_line(strokeDash=[4,2],color='#d62728').encode(alt.X('year'),alt.Y('PIR_index',title='PIR index',axis=alt.Axis(orient='right')),color=alt.value('#d62728'))
chart2=alt.layer(left,right).resolve_scale(y='independent').properties(title='Core Affordability Metric: PIR and PIR Index')
chart2.save('output/pir_main.png')
chart2


alt.LayerChart(...)

In [15]:
# Chart 3: Correlation heatmap
corr_cols=['home_price_index','mortgage_rate','income_median','permits','PIR','PTI','API']
corr=merged[corr_cols].corr(numeric_only=True)
corr_long=corr.reset_index().melt(id_vars='index').rename(columns={'index':'x','variable':'y'})
chart3=alt.Chart(corr_long).mark_rect().encode(x=alt.X('x',title=''),y=alt.Y('y',title=''),color=alt.Color('value',scale=alt.Scale(domainMid=0,scheme='redblue',reverse=True),title='corr')).properties(title='Correlation Heatmap',width=400,height=350)
chart3=chart3+alt.Chart(corr_long).mark_text().encode(x='x',y='y',text=alt.Text('value',format='.2f'),color=alt.value('black'))
chart3.save('output/correlation_heatmap.png')
chart3


alt.LayerChart(...)

In [17]:
hpi_chart=alt.Chart(merged).mark_line(color='blue').encode(alt.X('year'),alt.Y('home_price_index',title='Home Price Index'))
rate_chart=alt.Chart(merged).mark_line(color='red').encode(alt.X('year'),alt.Y('mortgage_rate',title='Mortgage Rate',axis=alt.Axis(orient='right')))
chart4=alt.layer(hpi_chart,rate_chart).resolve_scale(y='independent').properties(title='Home Price Index and Mortgage Rate Over Time')
chart4

alt.LayerChart(...)

In [19]:
# Chart 4: scatter with regression (mortgage rate vs PIR)
X=merged[['mortgage_rate']]
y=merged['PIR']
reg=LinearRegression().fit(X,y)
merged['pir_pred_by_rate']=reg.predict(X)

order=np.argsort(merged['mortgage_rate'].values)
line_df=merged.iloc[order][['mortgage_rate','pir_pred_by_rate']].copy()
merged5=merged[(merged['mortgage_rate']>=2)&(merged['mortgage_rate']<=9)]
line_df=line_df[(line_df['mortgage_rate']>=2)&(line_df['mortgage_rate']<=9)]
x_enc=alt.X('mortgage_rate',title='Mortgage Rate (%)',scale=alt.Scale(domain=[2,9]))
y_enc=alt.Y('PIR',title='PIR',scale=alt.Scale(domain=[0.2,0.6]))
scatter_chart=alt.Chart(merged5).mark_circle(size=40,opacity=0.8).encode(x_enc,y_enc)
line_chart=alt.Chart(line_df).mark_line(color='red').encode(x_enc,alt.Y('pir_pred_by_rate',scale=alt.Scale(domain=[0.2,0.6])))
chart5=alt.layer(scatter_chart,line_chart).properties(title='Mortgage Rate vs PIR',width=400,height=250)
chart5.save('output/pir_vs_rate_regression.png')
chart5

print('Slope:', reg.coef_[0])
print('Intercept:', reg.intercept_)
print('R^2:', reg.score(X,y))
chart5


Slope: 0.06680891455265922
Intercept: 0.05189775888844311
R^2: 0.7853600801073993


alt.LayerChart(...)

In [24]:
# Save updated annual table
merged.to_csv('output/merged_annual_updated.csv', index=False)
merged.tail()

,year,home_price_index,mortgage_rate,income_median,income_pc_real,permits,home_price_proxy,loan_amount,r_month,mortgage_payment_monthly,income_monthly,PIR,PTI,API,PIR_index,pir_pred_by_rate
20,2020,148.405477,3.111698,71243.0,60477.0,1256.116214,452470.449028,361976.359222,0.002593,1547.999273,5936.916667,0.260741,6.351086,0.000076,53.316759,0.259787
21,2021,164.771937,2.957692,72215.0,62167.0,1529.794856,502369.818168,401895.854534,0.002465,1685.252638,6017.916667,0.280039,6.956585,0.000078,57.262828,0.249498
22,2022,183.008395,5.344038,76744.0,59878.0,1433.657563,557970.584767,446376.467814,0.004453,2490.970746,6395.333333,0.389498,7.270543,0.000163,79.645160,0.408927
23,2023,192.599684,6.806731,80346.0,62786.0,1181.536624,587213.271905,469770.617524,0.005672,3064.659472,6695.500000,0.457719,7.308556,0.000209,93.595110,0.506648
24,2024,206.937817,6.721154,83327.0,NaN,1443.069769,630928.513242,504742.810594,0.005601,3264.079843,6943.916667,0.470063,7.571718,NaN,96.119216,0.500931


## Notes for interpretation
- **PIR/PTI** are the main affordability indicators.
- **API** is kept only as an auxiliary simplified pressure proxy.
- Anchored-price method supports trend interpretation, not absolute market valuation.
